# 신경망으로 챗봇 만들기

이전 노트북에서는 코사인 유사성을 가진 챗봇을 만드는 방법을 배웠습니다. 이제는 신경망을 이용해 어떻게 만들 수 있는지 살펴보겠습니다!

훈련 데이터를 만들고 신경망을 훈련시킨 다음 훈련된 모델을 사용하여 챗봇을 만들 것입니다.

먼저, 필수 라이브러리를 설치할 것입니다. 라이브러리가 설치되지 않은 경우에만 아래 몇 개의 블록을 주석 해제하십시오.

# 1. 라이브러리 설치

우선 이 신경망 구동 챗봇에 필요한 라이브러리를 import 하겠습니다.
Keras는 백엔드에서 텐서플로우(다른 하위 레벨 기계 학습 라이브러리) 를 활용하는 기계 학습 라이브러리입니다. 이렇게 하면 우리의 목적을 위해 심층 신경망을 쉽게 배포할 수 있습니다.

In [2]:
from keras.models import Sequential
from keras.losses import categorical_crossentropy
from keras.optimizers import SGD
from keras.layers import Dense
 
from numpy import argmax
import numpy as np
import re

# 2. 입력 훈련 데이터

먼저 챗봇에 대한 다음 교육 데이터를 포함하겠습니다.:
1. X는 사용자가 입력할 수 있는 다양한 입력을 나타냅니다.
2. Y는 입력의 의도를 나타냅니다.

In [3]:
X = ['Hi',
     'Hello',
     'How are you?',
     'I am making',
     'making',
     'working',
     'studying',
     'see you later',
     'bye',
     'goodbye']

In [4]:
print(len(X))

10


In [5]:
Y = ['greeting',
     'greeting',
     'greeting',
     'busy',
     'busy',
     'busy',
     'busy',
     'bye',
     'bye',
     'bye']

In [6]:
print(len(Y))

10


비슷한 의도를 가진 여러 개의 다른 문장들이 있다는 것을 주목하십시오. 여기에서는 3개의 의도(greeting, busy, bye)만 있지만 프로젝트에 원하는 만큼 추가할 수 있습니다.

이것은 챗봇이 작동하는 방식입니다:
1. 입력 문장으로부터, 우리는 훈련된 AI 모델을 사용하여 의도를 확인할 것이다.
2. 각 의도에 대해, 우리는 준비 된 응답을 가지고있다.

예를 들어, 입력의 의도가 인사말에 대한 것임을 확인하면 챗봇에 '안녕하세요'또는 '어떻게 지내십니까?'와 같은 인사말로 응답하도록 요청할 수 있습니다.

우리는 기계 학습을 사용하여 입력 문장을 다른 의도로 분류 할 수있는 모델을 만들 것입니다. 
다음과 같이 만듭니다:

1. 문장과 그 의도에 대한 목록을 포함하고 있는 훈련 데이터(위의 X및  Y)를 작성한다.
2. 훈련 데이터를 사용하여 분류기를 훈련한다. 
3. 입력 문장을 벡터화하고 분류기를 사용하여 의도를 결정한다. 

# 3. 텍스트 처리

평소와 같이 텍스트 처리부터 시작합니다. 그 과정을 기억하십니까?

## 3.1 알파벳과 숫자가 아닌 문자 제거

In [7]:
def remove_non_alpha_characters(sentence):
    new_sentence = ''
    for alphabet in sentence:
        if alphabet.isalpha() or alphabet == ' ':
            new_sentence += alphabet
    return new_sentence

In [8]:
remove_non_alpha_characters('Book #123 cAr!')

'Book  cAr'

위 함수를 정규표현식을 이용해서 다시 작성 합니다.

In [9]:
import re

def remove_non_alpha_characters2(sentence):
    # [^a-zA-Z\s] : 영문 알파벳(a-z, A-Z)과 공백(\s)이 아닌(^) 모든 문자를 매칭합니다.
    # 매칭된 문자들을 빈 문자열('')로 대체하여 제거합니다.
    return re.sub(r'[^a-zA-Z\s]', '', sentence)

# 예시 테스트
sample_text = "Hello! 123 World, @#$ Python"
print(remove_non_alpha_characters2(sample_text)) 
# 출력 결과: Hello  World  Python


Hello  World  Python


In [10]:
remove_non_alpha_characters2('Book #123 cAr!')

'Book  cAr'

In [11]:
def preprocess_data(X):
    X = [data_point.lower() for data_point in X]
    X = [remove_non_alpha_characters(
        sentence) for sentence in X]
    X = [data_point.strip() for data_point in X]
    X = [re.sub(' +', ' ',
                data_point) for data_point in X]
    return X

In [12]:
X = preprocess_data(X)

vocabulary = set()
for data_point in X:
    for word in data_point.split(' '):
        vocabulary.add(word)

vocabulary = list(vocabulary)

## 문서 벡터 생성

In [13]:
X_encoded = []

def encode_sentence(sentence):
    sentence = preprocess_data([sentence])[0]
    sentence_encoded = [0] * len(vocabulary)
    for i in range(len(vocabulary)):
        if vocabulary[i] in sentence.split(' '):
            sentence_encoded[i] = 1
    return sentence_encoded

X_encoded = [encode_sentence(sentence) for sentence in X]

In [14]:
classes = list(set(Y))

Y_encoded = []
for data_point in Y:
    data_point_encoded = [0] * len(classes)
    for i in range(len(classes)):
        if classes[i] == data_point:
            data_point_encoded[i] = 1
    Y_encoded.append(data_point_encoded)

# 4. 훈련 데이터 및 테스트 데이터 생성

In [15]:
X_train = X_encoded
y_train = Y_encoded
X_test = X_encoded
y_test = Y_encoded

훈련 및 테스트 데이터에 사용하는 데이터 출력 및 확인

In [16]:
print (y_test)

[[1, 0, 0], [1, 0, 0], [1, 0, 0], [0, 1, 0], [0, 1, 0], [0, 1, 0], [0, 1, 0], [0, 0, 1], [0, 0, 1], [0, 0, 1]]


In [17]:
print(len(X_train))

10


In [18]:
y_train

[[1, 0, 0],
 [1, 0, 0],
 [1, 0, 0],
 [0, 1, 0],
 [0, 1, 0],
 [0, 1, 0],
 [0, 1, 0],
 [0, 0, 1],
 [0, 0, 1],
 [0, 0, 1]]

y_train은 무엇을 의미합니까? 위에 표시된 배열을 이해합니까?

# 5. 모델 훈련

이제 훈련 데이터를 이용해 신경망을 훈련시키겠습니다.

In [19]:
model = Sequential()
model.add(Dense(units=64, activation='sigmoid',
                input_dim=len(X_train[0])))
model.add(Dense(units=len(y_train[0]), activation='softmax'))
model.compile(loss=categorical_crossentropy,
              optimizer=SGD(learning_rate=0.01,
                            momentum=0.9, nesterov=True))
model.fit(np.array(X_train), np.array(y_train), epochs=100, batch_size=16)


Epoch 1/100


c:\Users\User\anaconda3\Lib\site-packages\keras\src\layers\core\dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 444ms/step - loss: 1.1017
Epoch 2/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step - loss: 1.0977
Epoch 3/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - loss: 1.0932
Epoch 4/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - loss: 1.0888
Epoch 5/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - loss: 1.0851
Epoch 6/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - loss: 1.0824
Epoch 7/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - loss: 1.0805
Epoch 8/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - loss: 1.0792
Epoch 9/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step - loss: 1.0783
Epoch 10/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - loss: 1.0773
Epoch 11/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - loss: 1.0761
Epoch 12/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - loss: 1.0747
Epoch 13/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - loss: 1.0728
Epoch 14/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step - loss: 1.0707
Epoch 15/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step - loss: 1.0685
Epoch 16/100
1/1 ━━━━━━━━━━━━━

## 예측 목록 표시

In [20]:
# 해당 model을 이용해서 X_test 데이터를 추론한 후 그 결과를 predictions 변수에 저장합니다.
print(classes)
predictions = model.predict(np.array(X_test))


['greeting', 'busy', 'bye']
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step


# 모델 평가

이제 모델을 평가해 봅시다. 모델에 의한 예측과 테스트 데이터를 비교할 것입니다:

In [21]:
# 위에서 추론한 결과인 predictions 데이터가 y_test와 몇 개가 같은지 출력합니다.
predicted_classes = np.argmax(predictions, axis=1)
true_classes = np.argmax(np.array(y_test), axis=1)

correct = np.sum(predicted_classes == true_classes)
print(f"정답 개수: {correct} / {len(y_test)}")


정답 개수: 6 / 10


# 챗봇 테스트

이제 챗봇을 테스트해 보겠습니다! 문장을 입력한 다음 신경망에서 예측되는 클래스를 확인합니다:

In [22]:
while True:
    print("Enter a sentence (or 'quit' to stop)")
    sentence = input()
    
    if sentence.lower() in ['quit', 'exit', 'q']:
        print("Stopped.")
        break
    
    prediction = model.predict(np.array([encode_sentence(sentence)]))
    print(classes[np.argmax(prediction)])


Enter a sentence (or 'quit' to stop)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 71ms/step
busy
Enter a sentence (or 'quit' to stop)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step
busy
Enter a sentence (or 'quit' to stop)
Stopped.


챗봇을 멈출 수 없다는 것을 알고 있습니까? 나중에 종료 명령을 추가해야 합니다(이전 노트를 참조하여 수행 방법을 확인하십시오.).

일단은 위의 중지 버튼(인터럽트 버튼)을 눌러서 챗봇을 중지하면 됩니다.

시도해 보세요. 정지 버튼을 누르고 상자에 뭔가를 입력해 보세요.

# 도전과제

우리는 성공적으로 대화 의도에 우리의 입력을 매핑하는 신경망을 사용했습니다. 
여러분의 과제는 대화 의도를 챗봇이 말하는 특정 응답과 연결하는 것입니다. 
예를 들어, 만약 대화의 목적이 '인사' 라면, 여러분의 챗봇도 인사말을 하도록 하세요!

- 특정 키워드 종료
- time 기능 (현재 시간 출력)
- 날씨 (Beautiful Soup를 이용한 NAVER 날씨)
- 삼성전자 (Beautiful Soup를 이용한 삼성전자 주가)
- 영화순위 (Beautiful Soup를 이용한 영화정보)
- 가위, 바위, 보 중 한 단어를 입력하면 컴퓨터도 랜덤으로 하나를 선택해서 보여주고 결과를 알려준다

In [45]:
# your code here

Enter a sentence
hi
ah... all the best!
Enter a sentence
bye
Goodbye! See you next time!
Enter a sentence
busy
ah... all the best!
Enter a sentence
good morning
ah... all the best!
Enter a sentence
greeting
ah... all the best!
Enter a sentence
How are you?
Hi!
Enter a sentence


KeyboardInterrupt: Interrupted by user

### 잘했습니다! 신경망으로 간단한 챗봇을 성공적으로 만들었습니다! 챗봇을 어떻게 개선할 수 있을까요?
다음과 같은 방법으로 챗봇을 개선할 수 있습니다:
- 더 많은 교육 데이터 추가
- 더 많은 의도 추가
- 특정 주제에 초점을 맞추고 해당 주제에 많은 훈련 데이터로 챗봇 교육

### 출처:
https://blog.eduonix.com/internet-of-things/simple-nlp-based-chatbot-python/